In [20]:
import requests
from bs4 import BeautifulSoup
import re
import json
from urllib.parse import quote

In [ ]:
#  fetch from archived version if live page is not accessible
def wayback_fetch(url, headers):
    """Try to fetch the closest archived snapshot from the Wayback Machine."""
    api = f"https://archive.org/wayback/available?url={quote(url)}"
    r = requests.get(api, headers=headers, timeout=10)
    r.raise_for_status()
    data = r.json()
    snap = data.get("archived_snapshots", {}).get("closest")
    if not snap or not snap.get("available"):
        raise requests.exceptions.RequestException("No archived snapshot available")
    snapshot_url = snap.get("url")
    rs = requests.get(snapshot_url, headers=headers, timeout=15)
    rs.raise_for_status()
    return BeautifulSoup(rs.text, "html.parser")


def scrape_article(url):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    # fetch website
    soup = None
    try:
        res = requests.get(url, headers=headers, timeout=15)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, "html.parser")
    except requests.exceptions.RequestException:
        try:
            soup = wayback_fetch(url, headers)
        except requests.exceptions.RequestException:
            raise RuntimeError(f"Please try another link.")

    # try to detect JSON data (for sites like NBC, CNN, etc.)
    script_tag = soup.find("script", text=re.compile(r"window\.__data\s*="))
    if script_tag and script_tag.string:
        try:
            json_text = re.search(r"window\.__data\s*=\s*({.*});", script_tag.string).group(1)
            data = json.loads (json_text)
            # Try to find generic fields like headline/description/transcript
            title = data.get("initialState", {}).get("video", {}).get("currentVideo", {}).get("headline")
            body = (
                data.get("initialState", {}).get("video", {}).get("currentVideo", {}).get("description", "")
                + "\n"
                + data.get("initialState", {}).get("video", {}).get("currentVideo", {}).get("transcript", "")
            )
            if title and body:
                return title, clean_text(body)
        except Exception:
            pass

    # find title using common tags
    title_candidates = [
        soup.find("h1"),
        soup.find("meta", property="og:title"),
        soup.find("meta", attrs={"name": "twitter:title"}),
    ]
    title = next(
        (t.get_text(strip=True) if hasattr(t, "get_text") else t["content"])
        for t in title_candidates
        if t
    ) if any(title_candidates) else "(No title found)"

    # collect paragraphs from likely article sections
    body_candidates = [
        "article",
        "main",
        "div[class*='article']",
        "div[class*='content']",
        "div[itemprop='articleBody']",
        "section[name='articleBody']",
    ]

    paragraphs = []
    for selector in body_candidates:
        elements = soup.select(selector + " p")
        if elements:
            paragraphs = elements
            break

    if not paragraphs:
        paragraphs = soup.find_all("p")

    body_text = "\n".join(p.get_text(strip=True) for p in paragraphs)
    return title, clean_text(body_text)


def clean_text(text):
    """Remove irrelevant or boilerplate lines commonly found in news sites."""
    bad_phrases = [
        "news alerts",
        "breaking news",
        "there are no new alerts",
        "©",
        "sign up",
        "subscribe",
        "follow us",
        "advertisement",
        "read more",
        "all rights reserved",
        "supported by"
    ]

    cleaned_lines = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        lower_line = line.lower()
        if any(bad in lower_line for bad in bad_phrases):
            continue
        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)



📰 Title:
 Zohran Mamdani takes on governing as the left and right fight to define him

📜 Clean Article Body:
 Zohran Mamdani’s rapid rise from outer-borough state assemblyman to shock Democratic mayoral nominee was already one of the biggest political stories in years in New York City.
Now, the 34-year-old democratic socialist’s victory in Tuesday’s mayoral election has put him on the verge of becoming an even bigger story nationally. Operatives and political leaders on the progressive left and MAGA right are looking to nationalize Mamdani’s narrative for the midterm elections and beyond, as he begins his transition from campaigning to governing the nation’s largest city.
Whether the political tactic works for either side, to change or brand the Democratic Party, largely depends on just one thing: whether Mamdani is as effective a mayor as he was a candidate.
His success will depend on a few factors, starting with whom he hires and builds coalitions with, as well as which issues he pri

C:\Users\vewiz\AppData\Local\Temp\ipykernel_28352\4142041333.py:45: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  script_tag = soup.find("script", text=re.compile(r"window\.__data\s*="))


In [ ]:
# Example usage
if __name__ == "__main__":
    url = "https://www.nbcnews.com/politics/elections/zohran-mamdani-takes-governing-left-right-fight-define-rcna241705"
    title, body = scrape_article(url)

    print("Title:\n", title)
    print("\nClean Article Body:\n", body)